# Project Nano-Myo
## Notebook 04 - Compression


## Step 1 - Install


In [ ]:
!pip install tensorflow-model-optimization tf_keras imbalanced-learn --quiet


## Step 2 - Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 3 - Imports


In [ ]:
import json
import os
import sys
from collections import Counter
from pathlib import Path

if 'tensorflow' in sys.modules:
    raise RuntimeError('Restart runtime before Notebook 04 so TF_USE_LEGACY_KERAS is set before TensorFlow import.')

os.environ['TF_USE_LEGACY_KERAS'] = '1'

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import tensorflow_model_optimization as tfmot
from imblearn.over_sampling import SMOTE
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / 'Nano_Myo'
FEATURE_DIR = PROJECT_ROOT / 'features'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_SCHEMA_VERSION = 'e2_only_80feat_wl_ssc_rest25000_v1'
INPUT_DIM = 80
HIDDEN_UNITS = (64, 32)
NUM_CLASSES = 10
REST_CAP = 25000
ACTIVE_SMOTE_TARGET = 5000

FLOAT_MODEL_PATH = MODEL_DIR / 'rebalanced_float32.keras'
QAT_MODEL_PATH = MODEL_DIR / 'qat_model.keras'
TFLITE_MODEL_PATH = MODEL_DIR / 'nano_myo.tflite'
RESULTS_PATH = MODEL_DIR / 'compression_results.txt'


## Step 4 - Load Artifacts


In [ ]:
paths = {
    'X_train': FEATURE_DIR / f'X_train_{FEATURE_SCHEMA_VERSION}.npy',
    'y_train': FEATURE_DIR / f'y_train_{FEATURE_SCHEMA_VERSION}.npy',
    'X_test': FEATURE_DIR / f'X_test_{FEATURE_SCHEMA_VERSION}.npy',
    'y_test': FEATURE_DIR / f'y_test_{FEATURE_SCHEMA_VERSION}.npy',
    'metadata': FEATURE_DIR / f'nano_myo_features_holdout_s9_s10_{FEATURE_SCHEMA_VERSION}_metadata.json',
}

missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError('\n'.join(missing))

with open(paths['metadata'], 'r', encoding='utf-8') as f:
    metadata = json.load(f)

if metadata['feature_schema_version'] != FEATURE_SCHEMA_VERSION:
    raise ValueError(metadata['feature_schema_version'])
if metadata['exercise_used'] != 'E2':
    raise ValueError(metadata['exercise_used'])
if metadata['input_dim'] != INPUT_DIM:
    raise ValueError(metadata['input_dim'])

X_train = np.load(paths['X_train']).astype(np.float32)
y_train = np.load(paths['y_train']).astype(np.int64)
X_test = np.load(paths['X_test']).astype(np.float32)
y_test = np.load(paths['y_test']).astype(np.int64)
TARGET_LABELS = {int(key): value for key, value in metadata['target_labels'].items()}

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'Train counts: {Counter(y_train.tolist())}')
print(f'Test counts: {Counter(y_test.tolist())}')


## Step 5 - Balance Training Set


In [ ]:
rest_indices = np.flatnonzero(y_train == 0)
active_indices = np.flatnonzero(y_train != 0)

if rest_indices.shape[0] > REST_CAP:
    rng = np.random.default_rng(RANDOM_SEED)
    rest_indices = rng.choice(rest_indices, size=REST_CAP, replace=False)

balance_indices = np.sort(np.concatenate([active_indices, rest_indices]))
X_balanced = X_train[balance_indices]
y_balanced = y_train[balance_indices]

counts = Counter(y_balanced.tolist())
sampling_strategy = {
    class_id: ACTIVE_SMOTE_TARGET
    for class_id in range(1, NUM_CLASSES)
    if counts.get(class_id, 0) < ACTIVE_SMOTE_TARGET
}

if sampling_strategy:
    smote = SMOTE(
        sampling_strategy=sampling_strategy,
        random_state=RANDOM_SEED,
        k_neighbors=5,
    )
    X_balanced, y_balanced = smote.fit_resample(X_balanced, y_balanced)

X_balanced = X_balanced.astype(np.float32)
y_balanced = y_balanced.astype(np.int64)

print(f'Balanced X_train: {X_balanced.shape}')
print(f'Balanced counts: {Counter(y_balanced.tolist())}')


## Step 6 - Model


In [ ]:
def build_model(input_dim=INPUT_DIM, hidden_units=HIDDEN_UNITS, num_classes=NUM_CLASSES):
    inputs = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(hidden_units[0], activation='relu')(inputs)
    x = tf.keras.layers.Dense(hidden_units[1], activation='relu')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs)


float_model = build_model()
float_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

param_count = float_model.count_params()
float_model.summary()
print(f'Parameter count: {param_count:,}')


## Step 7 - Train Float32


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
    )
]

float_history = float_model.fit(
    X_balanced,
    y_balanced,
    validation_split=0.1,
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1,
)

float_loss, float_accuracy = float_model.evaluate(X_test, y_test, verbose=0)
float_pred = np.argmax(float_model.predict(X_test, batch_size=256, verbose=0), axis=1)

print(f'Float32 accuracy: {float_accuracy:.4f}')


## Step 8 - Float32 Confusion Matrix


In [ ]:
labels = list(TARGET_LABELS.keys())
display_labels = [TARGET_LABELS[key] for key in labels]
float_cm = confusion_matrix(y_test, float_pred, labels=labels, normalize='true')

fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(float_cm, display_labels=display_labels).plot(
    include_values=True,
    cmap='Blues',
    ax=ax,
    xticks_rotation=45,
    values_format='.2f',
)
plt.title('Rebalanced Float32 Confusion Matrix')
plt.tight_layout()
plt.show()


## Step 9 - Quantization-Aware Training


In [ ]:
inputs = tf.keras.Input(shape=(INPUT_DIM,))
x = tf.keras.layers.Dense(64, activation='relu')(inputs)
x = tf.keras.layers.Dense(32, activation='relu')(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
functional_model = tf.keras.Model(inputs, outputs)
functional_model.set_weights(float_model.get_weights())

qat_model = tfmot.quantization.keras.quantize_model(functional_model)
qat_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

qat_history = qat_model.fit(
    X_balanced,
    y_balanced,
    validation_split=0.1,
    epochs=15,
    batch_size=64,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
        )
    ],
    verbose=1,
)

qat_loss, qat_accuracy = qat_model.evaluate(X_test, y_test, verbose=0)
qat_pred = np.argmax(qat_model.predict(X_test, batch_size=256, verbose=0), axis=1)

print(f'QAT accuracy: {qat_accuracy:.4f}')


## Step 10 - Convert TFLite


In [ ]:
representative_indices = np.random.default_rng(RANDOM_SEED).choice(
    X_train.shape[0],
    size=min(1000, X_train.shape[0]),
    replace=False,
)
representative_data = X_train[representative_indices].astype(np.float32)


def representative_dataset():
    for row in representative_data:
        yield [row.reshape(1, INPUT_DIM).astype(np.float32)]


converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
TFLITE_MODEL_PATH.write_bytes(tflite_model)
tflite_size_bytes = TFLITE_MODEL_PATH.stat().st_size

print(f'TFLite size: {tflite_size_bytes:,} bytes')


## Step 11 - Verify TFLite


In [ ]:
def run_tflite_classifier(tflite_path, X):
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    input_index = input_details['index']
    output_index = output_details['index']
    input_scale, input_zero_point = input_details['quantization']
    output_scale, output_zero_point = output_details['quantization']
    predictions = []

    for row in X.astype(np.float32):
        sample = row.reshape(1, INPUT_DIM)
        if input_details['dtype'] in (np.int8, np.uint8):
            sample = np.round(sample / input_scale + input_zero_point)
            sample = np.clip(sample, np.iinfo(input_details['dtype']).min, np.iinfo(input_details['dtype']).max)
            sample = sample.astype(input_details['dtype'])
        else:
            sample = sample.astype(input_details['dtype'])

        interpreter.set_tensor(input_index, sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_index)
        if output_details['dtype'] in (np.int8, np.uint8):
            output = (output.astype(np.float32) - output_zero_point) * output_scale
        predictions.append(int(np.argmax(output, axis=1)[0]))

    return np.asarray(predictions, dtype=np.int64)


tflite_pred = run_tflite_classifier(TFLITE_MODEL_PATH, X_test)
tflite_accuracy = float(np.mean(tflite_pred == y_test))
prediction_match_rate = float(np.mean(tflite_pred == qat_pred))

tflite_per_class = {}
for class_id, class_name in TARGET_LABELS.items():
    mask = y_test == class_id
    tflite_per_class[class_id] = float(np.mean(tflite_pred[mask] == y_test[mask])) if np.any(mask) else float('nan')
    print(f'{class_id}: {class_name}: {tflite_per_class[class_id]:.4f}')

print(f'TFLite accuracy: {tflite_accuracy:.4f}')
print(f'QAT to TFLite prediction match rate: {prediction_match_rate:.4f}')


## Step 12 - Save


In [ ]:
float_model.save(FLOAT_MODEL_PATH)
qat_model.save(QAT_MODEL_PATH)

lines = [
    'Project Nano-Myo compression results',
    f'feature_schema_version: {FEATURE_SCHEMA_VERSION}',
    f'input_dim: {INPUT_DIM}',
    f'hidden_units: {HIDDEN_UNITS}',
    f'parameter_count: {param_count}',
    f'float32_accuracy: {float_accuracy:.6f}',
    f'qat_accuracy: {qat_accuracy:.6f}',
    f'tflite_accuracy: {tflite_accuracy:.6f}',
    f'qat_tflite_prediction_match_rate: {prediction_match_rate:.6f}',
    f'tflite_size_bytes: {tflite_size_bytes}',
    f'under_16kb: {tflite_size_bytes < 16 * 1024}',
    '',
    'tflite_per_class_accuracy:',
]

for class_id, class_name in TARGET_LABELS.items():
    lines.append(f'{class_id}: {class_name}: {tflite_per_class[class_id]:.6f}')

RESULTS_PATH.write_text('\n'.join(lines), encoding='utf-8')

print(f'Float32 model saved: {FLOAT_MODEL_PATH}')
print(f'QAT model saved: {QAT_MODEL_PATH}')
print(f'TFLite model saved: {TFLITE_MODEL_PATH}')
print(f'Results saved: {RESULTS_PATH}')
